# ML-11 — Capstone: Refresh / Content Opportunity Scoring

**Author:** Albert Michael Tigo  
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook consolidates the capstone question, data contract, baseline, model comparison, validation audit, action playbook, reproducibility checks and ML-12 communication outputs. Run it from the repository root in Colab or locally.

**Important:** the starter snapshot is cross-sectional. The 90-day features overlap the 30-day label window, so this is contemporaneous ranking/decision support—not a future-decline forecast.

## 1. Research question

Can observable content and search signals help an editor prioritize pages for refresh review? The unit is one pseudonymous content item. The output is a ranked queue with a score, reason codes and suggested action. A human editor verifies every recommendation; the model never publishes or rewrites content.

## 2. Data and safety contract

Use the anonymized starter CSV (30,000 rows × 44 columns; 32 pseudonymous clients). The reference target is `is_declining_label`, derived from `trend_direction == 'down'` (16,262 positives; 54.2%). IDs are for grouping only. Exclude `trend_direction` and `trend_pct` from features because they define/encode the target; exclude IDs and product-decision flags. Do not expose client names, domains, URLs, titles or raw queries.

**Temporal caveat:** 90-day totals overlap the 30-day period used to construct the label. This snapshot cannot support a clean future prediction claim.

## 3. Baseline

The ML-07 rule flags pages at least 180 days since update and with at least 500 impressions in the 90-day window. Score = stale × visible × impressions. Reason code: `stale_and_visible`; action: `review_refresh`. The saved baseline notebook reports 17 actionable pages out of 30,000.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'run_all.py').exists():
    raise RuntimeError('Run this notebook from the repository root (the folder containing scripts/run_all.py).')
print('Repository root:', ROOT)
print('Python:', sys.version)

## 4. Re-run the reference pipeline

This regenerates the baseline, model metrics, predictions, ranked queue and model report from the anonymized starter dataset. It may take a few minutes.

In [ ]:
result = subprocess.run([sys.executable, 'scripts/run_all.py'], cwd=ROOT, text=True, capture_output=True)
print(result.stdout[-12000:])
if result.returncode != 0:
    print(result.stderr[-12000:])
    raise RuntimeError(f'Pipeline failed with exit code {result.returncode}')
print('Pipeline completed successfully.')

## 5. Results (same held-out split)

Load the regenerated metrics. The target base rate must be shown next to ranking metrics. The reference report's saved headline values are included as a cross-check, not as a substitute for rerunning.

In [ ]:
metrics_path = ROOT / 'outputs' / 'model_results.json'
report_path = ROOT / 'outputs' / 'model_report.md'
if not metrics_path.exists():
    raise FileNotFoundError('Expected outputs/model_results.json after running the pipeline.')
metrics = json.loads(metrics_path.read_text())
print('Rows:', metrics['input_rows'])
print('Split:', metrics['split_strategy'])
print('Positive rate:', round(metrics['target_positive_rate'], 3))
print('Best model:', metrics['best_model']['name'])
print('\nModel comparison')
for name, m in metrics['models'].items():
    print(name, 'ROC AUC=', round(m['roc_auc'],3), 'AP=', round(m['average_precision'],3), 'P@50=', round(m['precision_at_50'],3), 'Recall=', round(m['recall'],3), 'F1=', round(m['f1'],3))
print('Baseline:', metrics['baseline'])
print('\nTop features:')
for item in metrics['best_model']['feature_importance_top'][:10]: print(item)

## 6. Leakage and validation audit

The reference script attempts a client holdout (20% of clients) and falls back to a stratified row split if the held-out client set lacks both classes. Confirm which split was actually used. The model feature list must not contain `trend_direction`, `trend_pct`, `is_declining_label`, `client_id` or `content_id`.

The more important caveat is temporal: the label is derived from last-30-day versus previous-30-day impressions, while several model inputs aggregate the same trailing 90-day period. This overlap means the current result is not a future forecast. A future-facing model needs features strictly before the label window using daily warehouse data.

In [ ]:
from scripts import ml_utils
forbidden = {'trend_direction','trend_pct','is_declining_label','client_id','content_id'}
configured = set(ml_utils.MODEL_NUMERIC_FEATURES) | set(ml_utils.MODEL_CATEGORICAL_FEATURES)
print('Forbidden feature overlap:', sorted(forbidden & configured))
assert not (forbidden & configured), 'Leakage/identifier field found in configured features.'
print('Configured feature count:', len(configured))
print('Split strategy:', metrics['split_strategy'])
print('NOTE: no temporal separation is possible from this single 90-day snapshot.')

## 7. Ranked recommendations and human review

Use `outputs/refresh_queue.csv` as a triage queue. Action labels map to manual checks:

- `refresh_and_review_ctr`: inspect snippet/title alignment and search intent.
- `refresh_and_review_engagement`: inspect usefulness, page experience and answer quality.
- `refresh`: check for substantive content gaps before editing.
- `expand_and_refresh`: verify whether additional coverage is warranted.
- `monitor`: defer immediate work and monitor signals.

Never infer real page identity from pseudonymous IDs. Never auto-publish, delete or rewrite from a score alone.

In [ ]:
queue_path = ROOT / 'outputs' / 'refresh_queue.csv'
import pandas as pd
queue = pd.read_csv(queue_path)
print('Queue rows:', len(queue))
print('Action counts:')
print(queue['suggested_action'].value_counts().to_string())
print('\nTop 10 review rows (pseudonymous IDs only):')
cols = [c for c in ['final_rank','final_refresh_score','confidence','suggested_action','final_reason_codes','impressions_90d','sessions_90d'] if c in queue.columns]
display(queue[cols].head(10))

## 8. Limitations and reproducibility

This target is a trend proxy, not a refresh-success label. The data is cross-sectional, the feature windows overlap the target window, and client holdout is not time holdout. The result is not causal and should not be described as predicting Google's ranking algorithm. The reference report's headline metrics are historical saved outputs; rerun the pipeline above to regenerate the metrics JSON and predictions.

**Re-run:** `pip install -r requirements.txt` then `python scripts/run_all.py`. Random seed: 42. Keep raw datasets and generated per-page queues out of Git.

## 9. Acknowledgments and data credit

Built on the FlyRank ML Internship anonymized starter dataset and teaching pipeline. Data credit: [FlyRank](https://flyrank.ai).

## ML-12 — Five-minute demo outline

1. 0:00–0:40 — editorial triage problem.
2. 0:40–1:20 — anonymized 30,000-row snapshot and data safety.
3. 1:20–2:10 — stale-and-visible baseline.
4. 2:10–3:20 — model comparison, client holdout and temporal-overlap caveat.
5. 3:20–4:20 — ranked queue, reason codes and human review.
6. 4:20–5:00 — limitations and next step: time-separated validation using daily warehouse data.

### Social-post cut

I built a content-refresh prioritization experiment using FlyRank's anonymized search dataset (30,000 pages). The saved evaluation report records Random Forest Precision@50 of 0.74 versus 0.24 for the rule baseline. The key lesson: time-window alignment matters. This single-snapshot result is decision support—not proof that refreshing content causes traffic recovery or a future-ranking forecast.

### Employer-facing summary

I built an explainable content-refresh prioritization pipeline on a 30,000-row anonymized search dataset, including a rule baseline, three classifiers, client-holdout evaluation, reason codes and a ranked editorial review queue. The committed pipeline report records Random Forest ROC AUC 0.750 and Precision@50 0.740, compared with baseline Precision@50 0.240. I documented temporal overlap and framed the output as human-reviewed decision support rather than a future-performance or causal claim.